In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

# Add parent directory to path to import src modules
sys.path.append(os.path.abspath('..'))

from src.data_processor import VectorSequenceProcessor
from src.engine import VectorLSTM

In [ ]:
# 1. Load the processed sequences
processor = VectorSequenceProcessor()
data_path = '../data/train_FD001.txt'

if os.path.exists(data_path):
    print("Processing sequences...")
    df = processor.load_data(data_path)
    df = processor.calculate_rul(df)
    df = processor.normalize_sensors(df, fit=True)
    
    X, y = processor.gen_sequences(df, window_size=50)
    
    # Creating an 80-20 validation split
    split_idx = int(0.8 * len(X))
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]
    
    print(f"[Data] Training Sequences: {X_train.shape}")
    print(f"[Data] Validation Sequences: {X_val.shape}")
else:
    print(f"⚠️ Data source not found at: {data_path}")

In [ ]:
# 2 & 3. Initialize the VectorLSTM model from src/engine.py 
# and verify Compilation using Adam / Mean Squared Error loss
vector_lstm_wrapper = VectorLSTM()
model = vector_lstm_wrapper.model

# Explicitly compiling matching the req just in case 
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# 4. Train with EarlyStopping (monitor=val_loss, epochs=50, batch_size=64)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Initiating AI Training Loop...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 5. Save the trained model to models/
out_path = '../models/vxp2_lstm_v1.h5'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

model.save(out_path)
print(f"✅ Model exported successfully to {out_path}")